# Experiment 2 (Part 1 of 2): The Anatomy of LoRA with the PEFT Library

**Course:** HAAI302COM - Artificial Intelligence Laboratory
**Experiment 2:** Fine-tuning a Pre-trained LLM using LoRA (Parameter-Efficient Fine-Tuning)

## Aim
To understand, before any training, exactly what Low-Rank Adaptation (LoRA) adds to a pre-trained language model: which layers it touches, how many parameters it trains, why the model's behaviour is unchanged at the start, and how an adapter is saved, switched off and merged.

## Objectives
1. Estimate why full fine-tuning of a 7B model does not fit on ordinary hardware.
2. Understand the low-rank idea with plain arithmetic and a small `torch` demo.
3. Find the layers of GPT-2 that LoRA can target.
4. Wrap GPT-2 with `peft.get_peft_model` and inspect the new `lora_A` / `lora_B` matrices.
5. Run parameter experiments on `r`, `lora_alpha` and `target_modules`.
6. Save an adapter, disable it, and merge it back into the base weights.

Part 2 (`02_LoRA_Fine_Tuning_Text_to_SQL.ipynb`) performs the actual fine-tuning and the before/after evaluation.

## Theory in one paragraph
Full fine-tuning updates every weight of the model, so you need memory for the weights, their gradients and the optimizer's bookkeeping, and you must store a complete copy of the model for every task. LoRA freezes the original weights. Next to a chosen weight matrix W it adds two thin matrices, A and B, whose product B x A has the same shape as W but a very small **rank** r. Only A and B are trained. The layer's output becomes: original output + (B x A applied to the input) x (alpha / r). Because B starts at zero, the model at step 0 behaves exactly like the original. After training, the tiny A and B can be shipped as a separate "adapter" file, or added into W permanently.

## 0. Setup

- `peft` (Parameter-Efficient Fine-Tuning) is Hugging Face's library for LoRA and related methods.
- We only need GPT-2 small here. Nothing is trained in this notebook, so it runs on any CPU in seconds.

In [1]:
# !pip install -q transformers peft torch pandas
import os
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
import transformers, huggingface_hub
transformers.logging.set_verbosity_error()
huggingface_hub.utils.logging.set_verbosity_error()

import tempfile
import torch
import pandas as pd
import peft
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
print("peft", peft.__version__, "| transformers", transformers.__version__)

peft 0.21.2 | transformers 5.16.1


## 1. Why full fine-tuning is expensive

A rough rule for training with the Adam optimizer in 32-bit floats: each parameter needs 4 bytes for the weight, 4 for its gradient and 8 for Adam's two running averages, so about **16 bytes per trained parameter**, before counting activations. Let us apply this rule to GPT-2 and Falcon-7B.

In [2]:
models = {"GPT-2 small": 124_439_808, "Falcon-7B": 6_921_720_704}
rows = []
for name, n in models.items():
    rows.append({"model": name, "parameters": f"{n:,}",
                 "weights only (GB, fp32)": round(n * 4 / 1e9, 1),
                 "full fine-tune (GB, ~16 bytes each)": round(n * 16 / 1e9, 1)})
pd.DataFrame(rows)

,model,parameters,"weights only (GB, fp32)","full fine-tune (GB, ~16 bytes each)"
0,GPT-2 small,"124,439,808",0.5,2.0
1,Falcon-7B,"6,921,720,704",27.7,110.7


**Takeaway.** Full fine-tuning of Falcon-7B needs on the order of 110 GB of GPU memory just for weights, gradients and optimizer state. A free Colab T4 has 15 GB. Even GPT-2 small needs about 2 GB. If we only train a tiny fraction of the parameters, the gradient and optimizer cost shrinks by the same fraction.

## 2. The low-rank idea in plain arithmetic

Take one weight matrix of GPT-2: the attention projection `c_attn`, which has shape **768 x 2304**.

- A full update to it is another 768 x 2304 matrix: 768 x 2304 = **1,769,472** numbers.
- LoRA with rank r = 8 instead learns A, which squeezes the 768 inputs down to 8 numbers, and B, which expands those 8 numbers back out to 2304 outputs: 768 x 8 + 8 x 2304 = **24,576** numbers.

The bet LoRA makes: the *change* needed to adapt a model to a new task is simple enough to be described by a few directions (a low rank), even though the original weight matrix itself is complex.

In [3]:
d_in, d_out = 768, 2304
pd.DataFrame([{"r": r, "LoRA numbers (d_in x r + r x d_out)": d_in * r + r * d_out,
               "share of full update": f"{(d_in * r + r * d_out) / (d_in * d_out):.2%}"}
              for r in [1, 4, 8, 16, 64, 256]])

,r,LoRA numbers (d_in x r + r x d_out),share of full update
0,1,3072,0.17%
1,4,12288,0.69%
2,8,24576,1.39%
3,16,49152,2.78%
4,64,196608,11.11%
5,256,786432,44.44%


**Takeaway.** At r = 8 we train under 1.5% of the numbers of a full update for this matrix. At r = 256 LoRA would cost more than a third of a full update and lose most of its benefit. Typical values are r = 4 to 64.

### 2.1 See the rank with your own eyes

Multiply a random 2304 x 8 matrix B by a random 8 x 768 matrix A. The product is a full-size 2304 x 768 matrix, but `torch.linalg.matrix_rank` shows that it only has rank 8: it only "moves" the input in 8 independent directions.

In [4]:
torch.manual_seed(0)
A = torch.randn(8, 768)        # down-projection: 768 -> 8
B = torch.randn(2304, 8)       # up-projection:   8 -> 2304
delta_W = B @ A                # same shape as the full weight update
print("delta_W shape:", tuple(delta_W.shape), "| rank:", int(torch.linalg.matrix_rank(delta_W)))
print("numbers stored as A and B:", A.numel() + B.numel(), "| numbers in delta_W:", delta_W.numel())

delta_W shape: (2304, 768) | rank: 8
numbers stored as A and B: 24576 | numbers in delta_W: 1769472


**Takeaway.** 1.7 million numbers, but fully described by 24,576. LoRA stores and trains only A and B, never the big product.

## 3. Which layers can LoRA target in GPT-2?

LoRA attaches to linear layers. In GPT-2 these are implemented as `Conv1D` modules (a historical name; mathematically they are linear layers with the weight stored transposed). Each of the 12 transformer blocks has four:

| Module | Role |
|---|---|
| `attn.c_attn` | produces query, key and value in one go |
| `attn.c_proj` | mixes the attention heads' outputs |
| `mlp.c_fc` | feed-forward expansion 768 -> 3072 |
| `mlp.c_proj` | feed-forward contraction 3072 -> 768 |

We list them from `model.named_modules()` for block 0.

In [5]:
tok = AutoTokenizer.from_pretrained("gpt2")
base = AutoModelForCausalLM.from_pretrained("gpt2").eval()
rows = [{"module name": n, "type": type(m).__name__, "weight shape": tuple(m.weight.shape), "parameters": m.weight.numel()}
        for n, m in base.named_modules() if n.startswith("transformer.h.0.") and hasattr(m, "weight") and m.weight.dim() == 2]
pd.DataFrame(rows)

,module name,type,weight shape,parameters
0,transformer.h.0.attn.c_attn,Conv1D,"(768, 2304)",1769472
1,transformer.h.0.attn.c_proj,Conv1D,"(768, 768)",589824
2,transformer.h.0.mlp.c_fc,Conv1D,"(768, 3072)",2359296
3,transformer.h.0.mlp.c_proj,Conv1D,"(3072, 768)",2359296


**Takeaway.** In block 0 the four `Conv1D` layers hold about 7 million parameters. These names (`c_attn`, `c_proj`, `c_fc`) are what we pass to `target_modules`. Every model family uses its own names: Falcon uses `query_key_value`, Llama and Mistral use `q_proj`, `k_proj`, `v_proj`, `o_proj`.

## 4. Wrapping GPT-2 with LoRA

`LoraConfig` arguments:

| Argument | Meaning |
|---|---|
| `r` | rank of A and B (how many directions the update may use) |
| `lora_alpha` | scaling numerator; the update is multiplied by `lora_alpha / r` |
| `target_modules` | which layers get an adapter |
| `lora_dropout` | dropout applied to the input of the LoRA branch during training |
| `bias` | `"none"` keeps all biases frozen |
| `task_type` | `"CAUSAL_LM"` tells PEFT this is a text-generation model |
| `fan_in_fan_out` | `True` for GPT-2's transposed `Conv1D` weights |

`get_peft_model(base, config)` freezes every original weight and inserts the LoRA branches. Note: it modifies the model object in place, so we load a fresh copy for the wrapped version.

In [6]:
config = LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"], lora_dropout=0.05,
                    bias="none", task_type="CAUSAL_LM", fan_in_fan_out=True)
lora_model = get_peft_model(AutoModelForCausalLM.from_pretrained("gpt2"), config)
print("training mode after wrapping:", lora_model.training)
lora_model.eval()                    # switch off lora_dropout for the inference checks below
lora_model.print_trainable_parameters()

training mode after wrapping: True
trainable params: 294,912 || all params: 124,734,720 || trainable%: 0.2364


**Takeaway.** 294,912 trainable parameters out of about 124.7 million, roughly a quarter of one percent. That is 12 blocks x 24,576, exactly the number computed by hand in Section 2.

Also notice `training mode after wrapping: True`. `get_peft_model` prepares the model for training, so `lora_dropout` is active and two identical forward passes would give slightly different outputs. Always call `.eval()` before comparing or generating.

### 4.1 Inspect the new matrices

`named_parameters()` now contains extra entries with `lora_A` and `lora_B` in their names. `requires_grad` tells us which tensors the optimizer would update.

In [7]:
rows = [{"parameter": n.replace("base_model.model.transformer.", ""), "shape": tuple(p.shape), "trainable": p.requires_grad}
        for n, p in lora_model.named_parameters() if ".h.0.attn.c_attn" in n]
pd.DataFrame(rows)

,parameter,shape,trainable
0,h.0.attn.c_attn.base_layer.weight,"(768, 2304)",False
1,h.0.attn.c_attn.base_layer.bias,"(2304,)",False
2,h.0.attn.c_attn.lora_A.default.weight,"(8, 768)",True
3,h.0.attn.c_attn.lora_B.default.weight,"(2304, 8)",True


**Takeaway.** For block 0 we now have the original `c_attn` weight and bias (frozen, `trainable=False`) plus `lora_A` of shape (8, 768) and `lora_B` of shape (2304, 8), both trainable.

### 4.2 Why the model is unchanged at step 0

PEFT initialises `lora_A` with small random numbers and `lora_B` with **zeros**. So B x A = 0 and the adapter adds nothing until training changes B. Let us check both facts.

In [8]:
layer = lora_model.base_model.model.transformer.h[0].attn.c_attn
A0, B0 = layer.lora_A["default"].weight, layer.lora_B["default"].weight
pd.DataFrame({"matrix": ["lora_A", "lora_B"], "mean |value|": [A0.abs().mean().item(), B0.abs().mean().item()],
              "all zeros?": [bool((A0 == 0).all()), bool((B0 == 0).all())]})

,matrix,mean |value|,all zeros?
0,lora_A,0.017998,False
1,lora_B,0.000000,True


In [9]:
ids = tok("LoRA adapters are small", return_tensors="pt")
with torch.no_grad():
    out_lora = lora_model(**ids).logits
    out_base = base(**ids).logits
print("max difference between LoRA model and base model logits:", (out_lora - out_base).abs().max().item())

max difference between LoRA model and base model logits: 0.0


**Takeaway.** The difference is exactly 0.0. Starting from the base model's behaviour is deliberate: training begins from a working model and only learns the *change*, rather than first destroying what the model already knows.

### 4.3 The scaling factor `lora_alpha / r`

The LoRA output is multiplied by `scaling = lora_alpha / r`. PEFT stores it on each layer. The common rule of thumb `lora_alpha = 2 x r` gives scaling 2. Keeping the ratio fixed means you can change `r` without retuning the learning rate much.

In [10]:
pd.DataFrame([{"r": r, "lora_alpha": a, "scaling (alpha / r)": a / r} for r, a in [(4, 8), (8, 16), (16, 32), (16, 16), (64, 16)]])

,r,lora_alpha,scaling (alpha / r)
0,4,8,2.00
1,8,16,2.00
2,16,32,2.00
3,16,16,1.00
4,64,16,0.25


In [11]:
print("scaling stored on the layer:", layer.scaling)

scaling stored on the layer: {'default': 2.0}


**Takeaway.** Our layer stores `{'default': 2.0}` because we used r=8, alpha=16. With r=64 and alpha=16 the scaling drops to 0.25, so each update step changes the model a quarter as much: a hidden reason why "bigger r" sometimes appears to learn more slowly.

## 5. Parameter experiment: `r` and `target_modules`

We count trainable parameters for several configurations. Counting does not need training, so this is instant. `count_trainable` builds a fresh wrapped model each time.

In [12]:
def count_trainable(r, targets):
    m = get_peft_model(AutoModelForCausalLM.from_pretrained("gpt2"),
                       LoraConfig(r=r, lora_alpha=2 * r, target_modules=targets, task_type="CAUSAL_LM", fan_in_fan_out=True))
    trainable, total = m.get_nb_trainable_parameters()
    return trainable, f"{trainable / total:.3%}"

In [13]:
experiments = [(r, ["c_attn"]) for r in [1, 4, 8, 16, 64]] + [(8, ["c_attn", "c_proj"]), (8, ["c_attn", "c_proj", "c_fc"])]
pd.DataFrame([{"r": r, "target_modules": ", ".join(t), "trainable": (c := count_trainable(r, t))[0], "share of model": c[1]}
              for r, t in experiments])

,r,target_modules,trainable,share of model
0,1,c_attn,36864,0.030%
1,4,c_attn,147456,0.118%
2,8,c_attn,294912,0.236%
3,16,c_attn,589824,0.472%
4,64,c_attn,2359296,1.861%
5,8,"c_attn, c_proj",811008,0.648%
6,8,"c_attn, c_proj, c_fc",1179648,0.939%


**Takeaway.**
- Trainable parameters grow **linearly** with `r`: r=16 is twice r=8.
- Note that `c_proj` matches *both* `attn.c_proj` and `mlp.c_proj` (PEFT matches by name suffix), which is why adding it more than doubles the count.
- Adding the MLP layers (`c_fc`) usually helps more on new-domain tasks than raising `r`, because the feed-forward layers store much of a model's factual and stylistic knowledge. Even with all four layer types, we train well under 1% of the model.

## 6. Pretend training: give the adapter non-zero weights

To see the adapter switching on and off without spending time on training, we fill `lora_B` with small random values. This mimics what training does: it moves B away from zero.

In [14]:
torch.manual_seed(1)
with torch.no_grad():
    for n, p in lora_model.named_parameters():
        if "lora_B" in n:
            p.normal_(0, 0.02)              # simulate a trained adapter
    out_adapter = lora_model(**ids).logits
    with lora_model.disable_adapter():      # temporarily bypass every LoRA branch
        out_disabled = lora_model(**ids).logits
pd.DataFrame({"comparison": ["adapter ON vs base", "adapter OFF vs base"],
              "max logit difference": [(out_adapter - out_base).abs().max().item(), (out_disabled - out_base).abs().max().item()]})

,comparison,max logit difference
0,adapter ON vs base,0.537022
1,adapter OFF vs base,0.000000


**Takeaway.** With the adapter on, the outputs differ from the base model; with `disable_adapter()` they are identical again. One base model in memory can therefore serve many tasks: load several adapters and switch between them per request.

## 7. Saving an adapter: how small is it?

`save_pretrained` on a PEFT model writes only the adapter weights plus a small `adapter_config.json`, never the frozen base model.

In [15]:
save_dir = tempfile.mkdtemp()
lora_model.save_pretrained(save_dir)
files = {f: os.path.getsize(os.path.join(save_dir, f)) for f in os.listdir(save_dir)}
pd.DataFrame({"file": list(files), "size (KB)": [round(s / 1024, 1) for s in files.values()]})

,file,size (KB)
0,adapter_config.json,1.0
1,adapter_model.safetensors,1155.0
2,README.md,5.0


In [16]:
full_mb = sum(p.numel() * p.element_size() for p in base.parameters()) / 1e6
adapter_mb = files["adapter_model.safetensors"] / 1e6
print(f"full GPT-2 weights: {full_mb:.0f} MB | adapter: {adapter_mb:.2f} MB | ratio: 1 to {full_mb / adapter_mb:.0f}")

full GPT-2 weights: 498 MB | adapter: 1.18 MB | ratio: 1 to 421


**Takeaway.** The adapter is about a megabyte, compared to about 500 MB for the base model. For Falcon-7B the same idea turns a 14 GB checkpoint into an adapter of a few tens of MB. Ten customers, ten tasks: one base model plus ten tiny files.

In [17]:
import json
adapter_cfg = json.load(open(os.path.join(save_dir, "adapter_config.json")))
{k: adapter_cfg[k] for k in ["base_model_name_or_path", "peft_type", "task_type", "r", "lora_alpha", "lora_dropout", "target_modules", "fan_in_fan_out"]}

{'base_model_name_or_path': 'gpt2',
 'peft_type': 'LORA',
 'task_type': 'CAUSAL_LM',
 'r': 8,
 'lora_alpha': 16,
 'lora_dropout': 0.05,
 'target_modules': ['c_attn'],
 'fan_in_fan_out': True}

**Takeaway.** The JSON file has many more keys (most are `null` options for other PEFT methods); these are the ones that matter. It records the base model name, `r`, `lora_alpha` and `target_modules`. That is everything `PeftModel.from_pretrained(base, save_dir)` needs to rebuild the adapter on top of a freshly loaded base model, which we do in Part 2.

## 8. Merging: removing the adapter overhead for deployment

During inference the LoRA branch adds an extra small matrix multiply in every targeted layer. `merge_and_unload()` computes W + (B x A) x scaling once, writes it into W, and returns a plain `transformers` model with no LoRA modules left. The outputs must be identical to the adapter version.

In [18]:
merged = lora_model.merge_and_unload()
with torch.no_grad():
    out_merged = merged(**ids).logits
print("type after merge:", type(merged).__name__)
print("any lora modules left?", any("lora" in n for n, _ in merged.named_modules()))
print("max difference merged vs adapter:", (out_merged - out_adapter).abs().max().item())

type after merge: GPT2LMHeadModel
any lora modules left? False
max difference merged vs adapter: 6.103515625e-05


**Takeaway.** After merging we have an ordinary `GPT2LMHeadModel` with no LoRA modules and the same outputs, up to tiny floating-point rounding (around 1e-4 or less). Trade-off: a merged model can no longer switch adapters, so merge only when one task is final.

## 9. Summary

| Question | Answer from this notebook |
|---|---|
| Trainable parameters, r=8 on `c_attn` | |
| Trainable share with all four layer types | |
| Logit difference at initialisation | |
| Adapter file size | |

## Viva questions
1. Why is `lora_B` initialised to zeros instead of random values?
2. Write the number of LoRA parameters for a 4096 x 4096 matrix at r = 16.
3. What does `lora_alpha` do, and why is `lora_alpha = 2r` a common choice?
4. Why does `target_modules=["c_proj"]` match two different layers in GPT-2?
5. When would you keep an adapter separate, and when would you merge it?